# 도메인 분류기 재학습 노트북

**실제 도면(dataset_raw) 기반 재학습 — 합성 데이터 완전 제거**

**설계 목표**
- 피처: 87-dim (feature_extractor.py 기준)
- 모델: XGBoost + Platt Scaling (CalibratedClassifierCV cv=3, method='sigmoid')
- 평가: Hold-out 20% 테스트셋 + StratifiedKFold(n_splits=5) CV
- 데이터: dataset_raw/ 실제 도면 435개만 사용 (합성 데이터 제외)

**출력 파일**
- `backend/api/classifier_models/domain_classifier.pkl`
- `backend/api/classifier_models/label_encoder.pkl`
- `backend/api/classifier_models/domain_classifier_meta.json`

## Step 0 · 환경 설정

In [ ]:
import sys, os, json, shutil, math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── 경로 ──────────────────────────────────────────────────────────────────
RAW_DIR   = PROJECT_ROOT / 'dataset_raw'   # ← 실제 도면 데이터만 사용
MODEL_DIR = PROJECT_ROOT / 'backend' / 'api' / 'classifier_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DOMAIN_LABELS = ['arch', 'elec', 'fire', 'pipe']

def domain_from_filename(p: Path) -> str | None:
    stem = p.stem.lower()
    for prefix in DOMAIN_LABELS:
        if stem.startswith(prefix):
            return prefix
    return None

import sklearn, xgboost
print(f'sklearn : {sklearn.__version__}  (필요: 1.8.0)')
print(f'xgboost : {xgboost.__version__}  (필요: 3.1.x)')
print(f'RAW_DIR : {RAW_DIR}')
print(f'MODEL_DIR: {MODEL_DIR}')
assert sklearn.__version__.startswith('1.8'), f'sklearn 버전 불일치: {sklearn.__version__} — ml_env로 커널 변경 필요'

## Step 1 · 데이터 파일 목록 수집 (실제 도면만)

In [ ]:
samples = []
for p in sorted(RAW_DIR.glob('*.json')):
    domain = domain_from_filename(p)
    if domain:
        samples.append({'path': p, 'domain': domain})

df_files = pd.DataFrame(samples)
print(f'총 샘플 수: {len(df_files)}')
print(df_files['domain'].value_counts())
print()
print('⚠️  합성 데이터 일절 사용하지 않음 — dataset_raw 실제 도면만 사용')

## Step 2 · Stratified 80/20 분할

In [ ]:
from sklearn.model_selection import train_test_split

paths  = [row.path for row in df_files.itertuples()]
labels = [row.domain for row in df_files.itertuples()]

paths_train, paths_test, y_train_raw, y_test_raw = train_test_split(
    paths, labels,
    test_size=0.20,
    stratify=labels,
    random_state=42,
)

print(f'학습셋: {len(paths_train)}개  분포: {Counter(y_train_raw)}')
print(f'테스트셋: {len(paths_test)}개  분포: {Counter(y_test_raw)}')
print()
print('✅ 테스트셋은 Step 10 최종 평가에서만 사용')

## Step 3 · 피처 추출 (학습셋만, 87-dim)

In [ ]:
from backend.services.agents.common.domain_classifier.feature_extractor import extract_features

X_train_list, y_train_collected, failed_train = [], [], []
for p, d in zip(paths_train, y_train_raw):
    try:
        cad_json = json.loads(Path(p).read_text(encoding='utf-8'))
        X_train_list.append(extract_features(cad_json))
        y_train_collected.append(d)
    except Exception as e:
        failed_train.append((str(p), str(e)))

X_train = np.stack(X_train_list)
y_train = np.array(y_train_collected)

print(f'X_train shape: {X_train.shape}')
print(f'클래스 분포: {Counter(y_train)}')
assert X_train.shape[1] == 87, f'피처 차원 오류: expected 87, got {X_train.shape[1]}'
print('피처 차원 검증 완료')
if failed_train:
    print(f'실패: {len(failed_train)}건')
    for f in failed_train[:5]: print(' ', f)

## Step 4 · EDA (실제 도면 기반)

In [ ]:
docs_dir = PROJECT_ROOT / 'docs' / '분류기'
docs_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
unique, counts = np.unique(y_train, return_counts=True)
axes[0].bar(unique, counts, color=['steelblue', 'orange', 'crimson', 'teal'])
axes[0].set_title('Class Distribution (Real Drawings Train Set)')
axes[0].set_ylabel('Sample Count')

feat_var = X_train.var(axis=0)
top30_idx = np.argsort(feat_var)[::-1][:30]
axes[1].bar(range(30), feat_var[top30_idx])
axes[1].set_title('Feature Variance Top-30 (Real Drawings)')
axes[1].set_xlabel('Feature Index (sorted)')

plt.tight_layout()
plt.savefig(docs_dir / 'eda_real_data.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 · 데이터 증강 (학습셋 내부만)

In [ ]:
rng = np.random.default_rng(42)

def augment_to_balance(X, y, target_per_class: int, noise_std: float = 0.02):
    X_aug, y_aug = list(X), list(y)
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        n_need = max(0, target_per_class - len(idx))
        if n_need == 0:
            continue
        chosen = rng.choice(idx, size=n_need, replace=True)
        noise  = rng.normal(0, noise_std, (n_need, X.shape[1])).astype(np.float32)
        X_aug.extend(X[chosen] + noise)
        y_aug.extend([cls] * n_need)
    return np.stack(X_aug), np.array(y_aug)

max_count = Counter(y_train).most_common(1)[0][1]
X_aug, y_aug = augment_to_balance(X_train, y_train, target_per_class=max_count)

print(f'증강 전: {X_train.shape}  분포: {Counter(y_train)}')
print(f'증강 후: {X_aug.shape}   분포: {Counter(y_aug)}')
print('⚠️  증강은 학습셋 내부에서만 수행')

## Step 6 · XGBoost + Platt Scaling 모델 정의

In [ ]:
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y_aug)

xgb_base = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric='mlogloss',
    random_state=42,
    tree_method='hist',
    n_jobs=-1,
)

model = CalibratedClassifierCV(xgb_base, cv=3, method='sigmoid')
print(f'클래스: {list(le.classes_)}')

## Step 7 · CV 평가 (학습셋만)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_aug, y_enc, cv=skf, scoring='accuracy', n_jobs=-1)

print(f'CV 정확도 (5-fold, 학습셋 내부): {cv_scores.round(4)}')
print(f'평균: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print('📌 이 수치는 학습셋 추정값입니다.')
print('   진짜 성능은 Step 10의 hold-out 테스트셋 결과입니다.')

## Step 8 · 최종 학습

In [ ]:
model.fit(X_aug, y_enc)
print('최종 학습 완료')

from sklearn.metrics import classification_report
train_pred = model.predict(X_aug)
train_acc  = (train_pred == y_enc).mean()
print(f'학습 정확도 (훈련셋 재평가 — 의미없는 참고용): {train_acc:.4f}')
print()
print(classification_report(y_enc, train_pred, target_names=le.classes_))

## Step 9 · 테스트셋 피처 추출

In [ ]:
X_test_list, y_test_collected, failed_test = [], [], []
for p, d in zip(paths_test, y_test_raw):
    try:
        cad_json = json.loads(Path(p).read_text(encoding='utf-8'))
        X_test_list.append(extract_features(cad_json))
        y_test_collected.append(d)
    except Exception as e:
        failed_test.append((str(p), str(e)))

X_test = np.stack(X_test_list)
y_test = le.transform(np.array(y_test_collected))

print(f'테스트셋: {X_test.shape}  분포: {Counter(y_test_collected)}')
if failed_test:
    print(f'실패: {len(failed_test)}건')
    for f in failed_test[:3]: print(' ', f)

## Step 10 · Hold-out 최종 평가 (진짜 성능)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_pred = model.predict(X_test)
test_acc  = (test_pred == y_test).mean()

print('=' * 60)
print(f'>>> Hold-out 테스트 정확도 (진짜 성능): {test_acc:.4f}  ({int(test_acc*len(y_test))}/{len(y_test)})')
print('=' * 60)
print()
print(classification_report(y_test, test_pred, target_names=le.classes_))
print()
print(f'CV 추정 (참고용):      {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Hold-out 실제 성능:   {test_acc:.4f}')
print(f'4-class 랜덤 baseline: 0.2500')

docs_final = PROJECT_ROOT / 'docs' / '분류기' / 'figures' / 'final'
docs_final.mkdir(parents=True, exist_ok=True)

cm = confusion_matrix(y_test, test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=ax, cmap='Blues')
ax.set_title(f'Hold-out Confusion Matrix (acc={test_acc:.3f})')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(docs_final / 'confusion_matrix_real.png', dpi=150, bbox_inches='tight')
plt.show()

wrong_idx = np.where(test_pred != y_test)[0]
print(f'\n오분류 {len(wrong_idx)}/{len(y_test)}건:')
for i in wrong_idx:
    fname = paths_test[i].name
    true_lbl = le.inverse_transform([y_test[i]])[0]
    pred_lbl = le.inverse_transform([test_pred[i]])[0]
    conf = model.predict_proba(X_test[i:i+1])[0].max()
    print(f'  {fname:40s}  실제={true_lbl}  예측={pred_lbl}  신뢰도={conf:.3f}')

## Step 11 · 모델 저장

In [ ]:
import joblib

model_path   = MODEL_DIR / 'domain_classifier.pkl'
encoder_path = MODEL_DIR / 'label_encoder.pkl'
meta_path    = MODEL_DIR / 'domain_classifier_meta.json'

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
for p in [model_path, encoder_path, meta_path]:
    if p.exists():
        backup = p.parent / f'{p.stem}.{ts}.bak{p.suffix}'
        shutil.copy2(p, backup)
        print(f'백업: {backup.name}')

joblib.dump(model, model_path)
joblib.dump(le,    encoder_path)

meta = {
    'model_name': 'XGBoost+PlattScaling',
    'classes': list(le.classes_),
    'n_features': int(X_train.shape[1]),
    'holdout_test_accuracy': float(test_acc),
    'cv_accuracy_mean': float(cv_scores.mean()),
    'cv_accuracy_std':  float(cv_scores.std()),
    'random_baseline_accuracy': 0.25,
    'data_source': 'dataset_raw (real drawings only)',
    'n_samples_real': int(len(df_files)),
    'n_samples_train_original': int(len(X_train)),
    'n_samples_train_augmented': int(len(X_aug)),
    'n_samples_test': int(len(X_test)),
    'train_test_split': '80/20 stratified',
    'trained_at': datetime.now().isoformat(),
    'sklearn_version': sklearn.__version__,
    'xgboost_version': xgboost.__version__,
    'xgb_params': {
        'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 3,
    },
    'platt_scaling': {'cv': 3, 'method': 'sigmoid'},
    'stratified_kfold_n_splits': 5,
}

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'\n저장 완료:')
print(f'  {model_path}')
print(f'  {encoder_path}')
print(f'  {meta_path}')
print()
print(f'진짜 성능 (hold-out): {test_acc:.4f}')
print(f'랜덤 기준선:          0.2500')

## Step 12 · 최종 검증 (DomainClassifier 래퍼)

In [ ]:
import importlib
import backend.services.agents.common.domain_classifier.classifier as clf_module
importlib.reload(clf_module)
from backend.services.agents.common.domain_classifier.classifier import DomainClassifier

clf = DomainClassifier(model_dir=MODEL_DIR)
assert clf.is_loaded, '모델 로드 실패!'
print(f'is_loaded: {clf.is_loaded}')
print(f'진짜 성능 (meta): {clf.meta.get("holdout_test_accuracy", "없음"):.4f}')

import random
random.seed(0)
for row in random.sample(list(df_files.itertuples()), 5):
    cad_json = json.loads(Path(row.path).read_text(encoding='utf-8'))
    pred  = clf.predict(cad_json)
    proba = clf.predict_proba(cad_json)
    conf  = max(proba.values())
    ok = '✅' if pred == row.domain else '❌'
    print(f'{ok} {Path(row.path).name:40s}  실제={row.domain}  예측={pred}  신뢰도={conf:.3f}')

print('\n래퍼 검증 완료')
print(f'진짜 성능은 Step 10의 hold-out 결과: {test_acc:.4f}')